# 02 — Exploratory Data Analysis, Patient Funnel & SLA Statistical Testing

## Objective
1. Reconcile the patient conversion funnel (`Inquiry -> Consultation -> Quote -> Treatment Completion`) and separate raw stage flags from strict sequential conversion.
2. Analyze drop-off stages, reasons, and acquisition channels.
3. Perform statistical hypothesis testing (Chi-square test of independence & Welch's two-sample t-test) on country-specific SLA breaches vs. consultation booking.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data'

from src.data_quality import load_csvs
from src.analytics import (
    add_sla_breach,
    funnel,
    compute_funnel_tables,
    compute_dropoff_tables,
    sla_test,
    format_sla_hypothesis_table,
    compute_country_sla,
)

In [2]:
tables = load_csvs(DATA)
pj = tables['patient_journey']
cr = tables['country_reference']

funnel_raw, funnel_strict = compute_funnel_tables(pj)
print('=== Raw Flag Funnel ===')
display(funnel_raw)
print('=== Strict Sequential Funnel (Requires quote_shared=1) ===')
display(funnel_strict)

=== Raw Flag Funnel ===
=== Strict Sequential Funnel (Requires quote_shared=1) ===


,stage,denominator,count,conversion_pct
0,Total Inquiries,1500,1500,100.000000
1,Consultations Booked,1500,984,65.600000
2,Quotes Shared,984,561,57.012195
3,Treatments Completed (recorded flag),561,554,98.752228


,stage,denominator,count,conversion_pct
0,Total Inquiries,1500,1500,100.000000
1,Consultations Booked,1500,984,65.600000
2,Quotes Shared,984,561,57.012195
3,Treatments Completed with Quote Flag,561,325,57.932264


## 1. Drop-Off Stage & Reason Breakdown

In [3]:
dropoff_stage, dropoff_source = compute_dropoff_tables(pj)
display(dropoff_stage)
display(dropoff_source)

reasons = (
    pj[pj['drop_off_flag'] == 1]
    .groupby('reason_for_drop_off', as_index=False)
    .agg(cases=('patient_id', 'count'), exposure_inr=('estimated_case_value_inr', 'sum'))
    .sort_values('cases', ascending=False)
)
display(reasons)

,drop_off_stage,cases,pct_of_coded_dropoffs
0,Inquiry,103,29.428571
1,First Response,89,25.428571
2,Consultation,55,15.714286
3,Treatment Plan,49,14.000000
4,Quote Shared,43,12.285714
5,Travel Planning,11,3.142857


,lead_source,dropoffs,inquiries,dropoff_rate,share_of_dropoffs
0,Chatbot,57,254,22.440945,16.285714
1,Email,45,238,18.907563,12.857143
2,Phone,62,244,25.409836,17.714286
3,Referral Partner,63,246,25.609756,18.000000
4,Web Form,67,246,27.235772,19.142857
5,WhatsApp,56,272,20.588235,16.000000


,reason_for_drop_off,cases,exposure_inr
7,No follow-up,66,33122591
1,Cost concern,62,27795105
8,Response delay,60,31328917
0,Chose competitor,45,20217298
9,Travel constraints,30,15867385
2,Documentation pending,26,12353891
3,Insurance/payment issue,24,10011073
6,Medical ineligibility,21,8548110
5,Low intent,13,5680913
4,Language barrier,3,1586023


## 2. Statistical Testing: Response-Time SLA Breach vs. Consultation Booking

In [4]:
sla_df = add_sla_breach(pj, cr)
stats = sla_test(sla_df)
display(format_sla_hypothesis_table(stats))
print(f"Cramer's V: {stats['cramers_v']:.4f} | Cohen's d: {stats['cohens_d']:.4f}")
print(f"95% CI for Consultation Rate Gap: [{stats['consultation_rate_gap_ci_95'][0]*100:.2f}%, {stats['consultation_rate_gap_ci_95'][1]*100:.2f}%]")

Cramer's V: 0.1311 | Cohen's d: -0.4424
95% CI for Consultation Rate Gap: [-22.35%, -11.12%]


,metric,value
0,SLA breach count,1.242000e+03
1,SLA breach rate,8.280000e-01
2,Consultation rate: no breach,7.945736e-01
3,Consultation rate: breach,6.272142e-01
4,Absolute consultation-rate gap,-1.673595e-01
5,Chi-square statistic,2.577835e+01
6,Chi-square p-value,3.829592e-07
7,Mean response time: booked,1.576555e+01
8,Mean response time: not booked,1.939981e+01
9,Welch t-statistic,-7.956887e+00


## 3. Country-Wise SLA & Conversion Performance

In [5]:
country_sla = compute_country_sla(pj, cr)
display(country_sla.sort_values('sla_breach_rate', ascending=False))

,country,leads,baseline_sla_hours,sla_breach_rate,consultation_rate,completion_rate
8,Sri Lanka,116,6,0.939655,0.681034,0.353448
5,Nigeria,117,6,0.931624,0.615385,0.282051
7,Philippines,141,8,0.914894,0.609929,0.368794
4,Nepal,133,6,0.902256,0.654135,0.353383
9,Tanzania,120,8,0.883333,0.700000,0.341667
3,Malaysia,130,8,0.853846,0.669231,0.415385
1,India,94,8,0.829787,0.670213,0.361702
11,UAE,143,8,0.797203,0.643357,0.391608
0,Bangladesh,136,12,0.750000,0.683824,0.375000
2,Kenya,126,12,0.738095,0.634921,0.444444
